# Loan Default Risk Prediction

Predicting whether a borrower will default on a loan, using the Credit Risk dataset from Kaggle (`credit_risk_dataset.csv`).
The notebook covers data cleaning, exploratory analysis, model comparison (Logistic Regression vs Random Forest), tuning, and a Streamlit app for scoring new applicants.

**Target:** `loan_status` (1 = default, 0 = no default)

## 1. Setup

In [ ]:
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

## 2. Data Loading

In [ ]:
df = pd.read_csv("credit_risk_dataset.csv")
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

| Column | Description |
|---|---|
| person_age | Age of the applicant |
| person_income | Annual income |
| person_home_ownership | RENT / OWN / MORTGAGE / OTHER |
| person_emp_length | Years of employment |
| loan_intent | Purpose of the loan |
| loan_grade | Loan grade (A best, G worst) |
| loan_amnt | Loan amount |
| loan_int_rate | Interest rate |
| loan_percent_income | Loan amount as a share of income |
| cb_person_default_on_file | Previous default on record (Y/N) |
| cb_person_cred_hist_length | Credit history length in years |
| loan_status | Target |

## 3. Target Distribution

In [ ]:
print(df["loan_status"].value_counts(normalize=True).mul(100).round(2))

ax = sns.countplot(x="loan_status", data=df, palette=["#4C9F70", "#D9534F"])
ax.set_title("Target distribution (0 = no default, 1 = default)")
plt.show()

Defaults are the minority class, so accuracy alone is not a reliable metric here. I use recall, F1 and ROC-AUC alongside it.

## 4. Data Quality

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0]
print(pd.DataFrame({"count": missing, "percent": (missing / len(df) * 100).round(2)}))
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["person_age", "person_emp_length", "person_income"]):
    sns.boxplot(y=df[col], ax=ax, color="#8ecae6")
    ax.set_title(col)
plt.tight_layout()
plt.show()

print("Age > 100:", (df["person_age"] > 100).sum())
print("Employment length > 60:", (df["person_emp_length"] > 60).sum())

## 5. Data Cleaning

In [ ]:
rows_before = len(df)

df = df.drop_duplicates()
# ages above 100 and employment lengths above 60 years are data entry errors
df = df[df["person_age"] <= 100]
df = df[df["person_emp_length"].isna() | (df["person_emp_length"] <= 60)]
df = df.reset_index(drop=True)

print(f"Rows: {rows_before} -> {len(df)}")

Missing values in `person_emp_length` and `loan_int_rate` are not filled here. They are imputed inside the modelling pipeline (section 9) so that the test set does not influence the imputation values.

## 6. Exploratory Data Analysis

In [ ]:
num_cols = df.select_dtypes(include="number").drop(columns="loan_status").columns.tolist()
cat_cols = df.select_dtypes(exclude="number").columns.tolist()

n = len(num_cols)
rows = int(np.ceil(n / 3))

fig, axes = plt.subplots(rows, 3, figsize=(15, 4 * rows))
for ax, col in zip(axes.flat, num_cols):
    sns.histplot(df[col], kde=True, ax=ax, color="#277da1")
    ax.set_title(col)
for ax in list(axes.flat)[n:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col in zip(axes.flat, cat_cols):
    sns.countplot(x=col, data=df, order=df[col].value_counts().index, ax=ax, palette="viridis")
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col in zip(axes.flat, cat_cols):
    rate = df.groupby(col)["loan_status"].mean().mul(100).sort_values(ascending=False)
    sns.barplot(x=rate.index, y=rate.values, ax=ax, palette="rocket")
    ax.set_title(f"Default rate (%) by {col}")
    ax.tick_params(axis="x", rotation=30)
    for i, v in enumerate(rate.values):
        ax.text(i, v + 0.5, f"{v:.1f}", ha="center")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(rows, 3, figsize=(15, 4 * rows))
for ax, col in zip(axes.flat, num_cols):
    sns.boxplot(x="loan_status", y=col, data=df, ax=ax, showfliers=False, palette=["#4C9F70", "#D9534F"])
    ax.set_title(f"{col} by loan_status")
for ax in list(axes.flat)[n:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rate_bucket = pd.cut(df["loan_int_rate"], bins=[0, 8, 10, 12, 14, 16, 30])
df.groupby(rate_bucket)["loan_status"].mean().mul(100).plot(kind="bar", ax=axes[0], color="#e76f51")
axes[0].set_title("Default rate (%) by interest rate")

lpi_bucket = pd.cut(df["loan_percent_income"], bins=[0, 0.1, 0.2, 0.3, 0.4, 1.0])
df.groupby(lpi_bucket)["loan_status"].mean().mul(100).plot(kind="bar", ax=axes[1], color="#f4a261")
axes[1].set_title("Default rate (%) by loan as share of income")

plt.tight_layout()
plt.show()

In [ ]:
corr = df[num_cols + ["loan_status"]].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix")
plt.show()

**Observations**
- Default rate rises steadily from loan grade A to G.
- Applicants who rent default more often than those who own or have a mortgage.
- A previous default on file is associated with a clearly higher default rate.
- Default rate increases with interest rate and with loan amount relative to income.
- `loan_percent_income` and `loan_int_rate` show the strongest correlation with the target.

## 7. Feature Engineering

In [ ]:
# share of the applicant's life spent with a credit record
df["cred_hist_to_age"] = df["cb_person_cred_hist_length"] / df["person_age"]

## 8. Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns="loan_status")
y = df["loan_status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(X_train.shape, X_test.shape)

## 9. Preprocessing Pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_features = X_train.select_dtypes(include="number").columns.tolist()
cat_features = X_train.select_dtypes(exclude="number").columns.tolist()

# imputation and scaling sit inside the pipeline so they are fit on training data only
preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())]), num_features),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_features),
])

## 10. Models and Cross-Validation

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

# class_weight="balanced" compensates for the minority default class
log_reg = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])
rf = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                     random_state=42, n_jobs=-1)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

cv_rows = {}
for name, model in [("Logistic Regression", log_reg), ("Random Forest", rf)]:
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    cv_rows[name] = {m: scores[f"test_{m}"].mean() for m in scoring}

pd.DataFrame(cv_rows).T.round(4)

## 11. Test Set Evaluation

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

log_reg.fit(X_train, y_train)
rf.fit(X_train, y_train)

def evaluate(name, model, threshold=0.5):
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= threshold).astype(int)
    return {"Model": name,
            "Accuracy": accuracy_score(y_test, pred),
            "Precision": precision_score(y_test, pred),
            "Recall": recall_score(y_test, pred),
            "F1": f1_score(y_test, pred),
            "ROC-AUC": roc_auc_score(y_test, proba)}

results = pd.DataFrame([evaluate("Logistic Regression", log_reg),
                        evaluate("Random Forest", rf)]).set_index("Model").round(4)
results

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, classification_report

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, model) in zip(axes[:2], [("Logistic Regression", log_reg), ("Random Forest", rf)]):
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)

RocCurveDisplay.from_estimator(log_reg, X_test, y_test, ax=axes[2], name="Logistic Regression")
RocCurveDisplay.from_estimator(rf, X_test, y_test, ax=axes[2], name="Random Forest")
axes[2].plot([0, 1], [0, 1], "k--")
axes[2].set_title("ROC curve")
plt.tight_layout()
plt.show()

print(classification_report(y_test, rf.predict(X_test), target_names=["No default", "Default"]))

## 12. Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "model__n_estimators": [200, 300, 400],
    "model__max_depth": [10, 15, 20, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2"],
}

search = RandomizedSearchCV(rf, param_dist, n_iter=10, cv=3, scoring="roc_auc",
                            random_state=42, n_jobs=-1)
search.fit(X_train, y_train)

best_rf = search.best_estimator_
print(search.best_params_)
print("CV ROC-AUC:", round(search.best_score_, 4))

In [ ]:
tuned = pd.DataFrame([evaluate("Random Forest (tuned)", best_rf)]).set_index("Model").round(4)
final_results = pd.concat([results, tuned])
final_results

## 13. Feature Importance

In [ ]:
names = [f.split("__", 1)[1] for f in best_rf.named_steps["prep"].get_feature_names_out()]
importances = pd.Series(best_rf.named_steps["model"].feature_importances_, index=names)
top15 = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(9, 6))
sns.barplot(x=top15.values, y=top15.index, palette="viridis")
plt.title("Top 15 features")
plt.xlabel("Importance")
plt.show()

## 14. Decision Threshold

A missed default costs more than a rejected good applicant, so I checked how precision and recall change as the threshold moves away from 0.5.

In [ ]:
proba = best_rf.predict_proba(X_test)[:, 1]
rows_t = []
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    pred = (proba >= t).astype(int)
    rows_t.append({"Threshold": t,
                   "Precision": precision_score(y_test, pred),
                   "Recall": recall_score(y_test, pred),
                   "F1": f1_score(y_test, pred)})
pd.DataFrame(rows_t).set_index("Threshold").round(3)

## 15. Save Model

In [ ]:
joblib.dump(best_rf, "model.pkl")

## 16. Streamlit App
Run with `streamlit run app.py` after saving the model.

In [ ]:
%%writefile app.py
import joblib
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Loan Default Risk")
model = joblib.load("model.pkl")

st.title("Loan Default Risk Scoring")

col1, col2 = st.columns(2)
with col1:
    age = st.number_input("Age", 18, 100, 30)
    income = st.number_input("Annual income", 1000, 5000000, 50000, step=1000)
    home = st.selectbox("Home ownership", ["RENT", "OWN", "MORTGAGE", "OTHER"])
    emp = st.number_input("Employment length (years)", 0.0, 60.0, 3.0)
    intent = st.selectbox("Loan intent", ["EDUCATION", "MEDICAL", "VENTURE",
                                          "PERSONAL", "DEBTCONSOLIDATION", "HOMEIMPROVEMENT"])
with col2:
    grade = st.selectbox("Loan grade", list("ABCDEFG"))
    amount = st.number_input("Loan amount", 500, 100000, 10000, step=500)
    rate = st.number_input("Interest rate (%)", 1.0, 30.0, 11.0)
    default_file = st.selectbox("Previous default on file", ["N", "Y"])
    hist = st.number_input("Credit history length (years)", 1, 40, 4)

if st.button("Check risk"):
    row = pd.DataFrame([{
        "person_age": age,
        "person_income": income,
        "person_home_ownership": home,
        "person_emp_length": emp,
        "loan_intent": intent,
        "loan_grade": grade,
        "loan_amnt": amount,
        "loan_int_rate": rate,
        "loan_percent_income": amount / income,
        "cb_person_default_on_file": default_file,
        "cb_person_cred_hist_length": hist,
        "cred_hist_to_age": hist / age,
    }])
    prob = model.predict_proba(row)[0][1]
    st.metric("Default probability", f"{prob:.1%}")
    if prob >= 0.5:
        st.error("High risk")
    else:
        st.success("Low risk")

## 17. Results Summary

In [ ]:
print("Records used:", len(df))
print()
print(final_results)
print()
print("Top 5 features:")
print(top15.head(5).round(3).to_string())

## Key Design Decisions
- **No data leakage:** imputation, scaling and encoding live inside a scikit-learn `Pipeline`, so they are fit on training folds only, including during cross-validation.
- **Class imbalance:** handled with `class_weight="balanced"`, and models are compared on recall, F1 and ROC-AUC, not just accuracy.
- **Cleaning:** removed impossible values (age above 100, employment length above 60 years) instead of capping them, since they are entry errors rather than genuine extremes.
- **Engineered feature:** `cred_hist_to_age` captures how long an applicant has had credit relative to their age.
- **Threshold analysis:** checked how recall and precision trade off, since a missed default is costlier than a rejected good applicant.